# Week 1 — Statistical Thinking, Probability and the Generative View

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../01_Intro/Intro.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Distinguish a physical population, a realised sample, an observation, and an estimator.
- Use probability distributions, transformations, expectation, variance, covariance, and conditional probability in astronomical examples.
- Separate intrinsic population scatter from measurement uncertainty.
- Represent the binary-SMBH simulation as a generative model with latent physical parameters, nuisance/context variables, and observables.
- Recognise when a feature in a simulated catalogue is a prior/design choice rather than an empirical property of the Universe.

## Provider-confirmed dataset metadata

The simulation providers' reading notebook specifies the following quantities
explicitly:

- total binary mass: \(M_{\rm tot}\) in **solar masses**;
- mass ratio: \(q=M_2/M_1\);
- period: **observed orbital period in years**;
- eccentricity: dimensionless;
- inclination and argument of pericenter: **degrees**;
- RA/Dec: **degrees**;
- field: **WFD or DDF**;
- light-curve observation time: **MJD**;
- observed magnitude and error: **mag**.

The full with-inputs dataset is documented as containing **1,200,000
simulations**. The provider's blinded dataset excludes both `inputs` and
per-band `info`, which makes those variables legitimate hidden truths for
inverse-inference exercises.

The magnitude convention itself (e.g. AB) is not stated in the supplied reading
notebook and should not be assumed.


## Source-scope rule

For this course, the supplied files are the complete source of truth for the
simulation metadata. If a prior, convention, or physical prescription is not
documented, label it **unknown**.

This leads to an important statistical distinction:

\[
\text{empirical distribution in simulations}
\neq
\text{documented generating prior}
\neq
\text{cosmic population distribution}.
\]

We may measure the first directly. We should only claim the second if the
supplied material states it. The third requires inference from real observations
and is not established by this simulated catalogue alone.


## 1. Why astronomy needs statistical thinking


Astronomical inference is usually an **inverse problem**. We do not manipulate the source; we receive photons that have passed through an instrument, a cadence, a detection pipeline and a selection function. The parameter we care about may therefore be several modelling layers away from the recorded quantity. A useful habit is to separate four objects from the beginning:

1. the **population** we want to learn about;
2. the **latent state** of one astrophysical system;
3. the **observable data** generated from that state;
4. the **analysis procedure** that turns the data into an estimate.

For the binary-SMBH simulations, a schematic generative chain is

\[
\Lambda \rightarrow \theta_i \rightarrow x_i^{\rm ideal}
\rightarrow x_i^{\rm survey} \rightarrow D_i,
\]

where \(\Lambda\) represents population-level assumptions, \(\theta_i\) the physical parameters of system \(i\), \(x_i^{\rm ideal}\) an idealised variability signal, and \(x_i^{\rm survey}\) the irregular noisy multiband light curve.

This distinction matters immediately. The histogram of `inputs__mass_ratio` in the simulation is not automatically a measurement of the cosmic mass-ratio distribution. It may be close to the **simulation prior**. The statistical problem changes depending on whether the question is "what did we simulate?", "what would a survey detect?", or "what does the real Universe contain?"

Modern astrostatistics is therefore broader than choosing a regression algorithm. It includes the sampling model, the observation process, selection, computation, uncertainty and model criticism.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Load the teaching catalogue

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.append('../src')
from course_utils import load_scalar_sample

df = load_scalar_sample()
print(df.shape)
display(df.head())
display(df.dtypes.to_frame('dtype'))

## 2. Probability as a language for uncertainty


A random variable \(X\) is described by a probability distribution. For a continuous variable,

\[
P(a<X<b)=\int_a^b p(x)\,dx,\qquad \int p(x)\,dx=1.
\]

The expectation and variance are

\[
E[X]=\int x p(x)\,dx,\qquad
{\rm Var}(X)=E[(X-E[X])^2].
\]

For two variables, covariance and correlation quantify different ideas:

\[
{\rm Cov}(X,Y)=E[(X-E[X])(Y-E[Y])],
\]

\[
\rho_{XY}=\frac{{\rm Cov}(X,Y)}{\sigma_X\sigma_Y}.
\]

Correlation is useful but not causal. In a simulator, correlations can be produced by the prior, by deterministic transformations, by a survey selection function, or by the underlying physics. Students should always ask which path in the generative graph could produce the association.

Conditional probability is central:

\[
p(x,y)=p(x\mid y)p(y),
\]

and Bayes' theorem will later reverse this direction. Already in Week 1, the distinction between \(p(q\mid z)\) and \(p(z\mid q)\) should be explicit.

Astronomical variables often require transformations. If \(Y=\log_{10} X\), a distribution that is extremely skewed in \(X\) may be approximately regular in \(Y\). The sample supplied with this course illustrates this dramatically for total mass: the arithmetic mean is much larger than the median. This is not a numerical nuisance; it tells us that a summary on the linear scale can be scientifically misleading.

### Worked computational demonstration — Linear versus logarithmic mass

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
m = df['inputs__total_mass'].astype(float)
print('mean   =', m.mean())
print('median =', m.median())
print('mean log10 M =', np.log10(m).mean())

fig, ax = plt.subplots()
ax.hist(np.log10(m), bins=40)
ax.set_xlabel(r'$\log_{10}(M_{\rm tot})$')
ax.set_ylabel('count')
plt.show()

## 3. Population scatter versus measurement uncertainty


Suppose a population of quasars has an intrinsic distribution of a quantity \(X\), but each measurement is noisy:

\[
X_i^{\rm obs}=X_i^{\rm true}+\epsilon_i,\qquad
\epsilon_i\sim N(0,\sigma_i^2).
\]

The dispersion of the observed sample combines **population variation** and **measurement variation**. Treating the observed variance as if it were entirely intrinsic biases population inference.

The same principle applies to time-domain data. In the binary-SMBH file the magnitude scatter of a light curve includes genuine simulated variability and heteroscedastic photometric error. The columns `mag err` are therefore part of the likelihood, not decoration.

A simple generative statement is usually more informative than a long list of statistical tests:

\[
\theta_i\sim p(\theta\mid\Lambda),\quad
x_i^{\rm true}\sim p(x\mid\theta_i),\quad
x_i^{\rm obs}\sim p(x_{\rm obs}\mid x_i^{\rm true},\sigma_i).
\]

Every later week in this course will add detail to this chain.

### Worked computational demonstration — Monte-Carlo error propagation demo

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
rng = np.random.default_rng(2026)
x_obs, sigma = 10.0, 1.5
draws = rng.normal(x_obs, sigma, 100_000)
y = np.log10(draws[draws > 0])
print('median log10(x):', np.median(y))
print('68% interval:', np.quantile(y,[0.16,0.84]))

## 4. Anatomy of the binary-SMBH simulator


The scalar simulation inputs include total mass, mass ratio, period, eccentricity, inclination, argument of pericentre, redshift and Eddington fraction, together with discrete physical switches such as Doppler boosting, lensing flare and accretion modulation. The heavy part of the Parquet file contains irregular \(ugrizy\) time arrays, observed magnitudes and magnitude uncertainties. Per-band DRW/DHO truth parameters are also stored.

It is useful to classify variables into:

- **scientific targets / latent parameters**: quantities we may want to infer;
- **context or nuisance variables**: e.g. field and sky location, which can affect cadence;
- **simulation mechanism labels**: switches that define the forward model;
- **observables**: what a survey would provide;
- **ground-truth diagnostic quantities**: values available only because the data are simulated.

This classification prevents information leakage later. If the task is to infer period from the light curve, `inputs__period` cannot appear in the predictors. If the task is to recover a DRW timescale, the stored true `DRW tau` is an answer key, not a feature.

## 5. From descriptive plots to scientific questions


Descriptive statistics are useful only when tied to a scientific question. A histogram answers "what values are present in this sample?" It does not by itself answer "what distribution generated the Universe?" A scatter plot shows association but does not identify mechanism.

For every analysis in this course, write five statements before coding:

1. **Estimand:** what quantity or population property do I want?
2. **Observed data:** what is actually measured?
3. **Generative assumptions:** how could the data arise from the estimand?
4. **Uncertainty/selection:** what can distort the relation?
5. **Validation:** what observation or simulation check could falsify my conclusion?

This structure is intentionally repeated in labs and homework. The goal is to make it automatic by the end of the semester.

## Deep dive — factorising the simulator as a probability model

A useful way to convert a simulation into a statistical model is to write a factorisation of its joint distribution. One possible schematic factorisation is

\[
p(\Lambda,\theta,c,s,x_{\rm true},x_{\rm obs})
=
p(\Lambda)\,
p(\theta\mid\Lambda)\,
p(c)\,
p(s\mid\theta,c)\,
p(x_{\rm true}\mid\theta,s)\,
p(x_{\rm obs}\mid x_{\rm true},c).
\]

Here \(\theta\) denotes physical binary parameters, \(c\) observing context (field, sky position, cadence), \(s\) simulator mechanism/stochastic-process variables, and \(x\) the light-curve data. The exact Crete simulator may not factorise in precisely this form; the point is that a proposed factorisation forces us to ask what is generated from what.

This exercise exposes several common mistakes. First, conditioning on a descendant can induce associations between otherwise independent variables. Second, selecting on a detection event can change marginal distributions. Third, deterministic derived quantities can appear highly correlated without representing independent information. Finally, a simulator prior may be broad for computational coverage rather than astrophysical realism.

### Worked reasoning example

Suppose the simulator samples \(q\) uniformly but the detectability of periodic variability is larger at high \(q\). An observed detected sample will then have

\[
p(q\mid{\rm detected})
\propto p({\rm detected}\mid q)p(q),
\]

which is not uniform. A histogram from the detected catalogue therefore reflects both the simulation design and the selection process.

### Transformations require physical thought

Not every variable should be treated as Euclidean. Total mass and period span orders of magnitude, so log transforms are natural. Argument of pericentre is circular: \(1^\circ\) and \(359^\circ\) are close, even though their arithmetic difference is \(358^\circ\). Inclination conventions must be confirmed before constructing distances. Binary switches are categorical, not continuous.

A robust first analysis therefore includes a **variable-type audit** before computing correlations. Pearson correlation between a circular angle and a linear variable can be meaningless. A good analyst chooses a representation that reflects the parameter geometry.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Feigelson & Babu, *Modern Statistical Methods for Astronomy*.
- Ivezić et al., *Statistics, Data Mining, and Machine Learning in Astronomy*.
- Crete `01_Intro/Intro.ipynb`.

# Part II — Extended lecture development

The first part established probability language and the generative view. This
second part develops the ideas far enough that they can support a full
three-hour Master's lecture.

## 11. Probability factorisation as a modelling tool

The chain rule allows any joint distribution to be factorised:

\[
p(x_1,\ldots,x_n)
=
p(x_1)\,
p(x_2\mid x_1)\cdots
p(x_n\mid x_1,\ldots,x_{n-1}).
\]

The useful factorisation is the one that reflects the science. For a simplified
SMBH light-curve problem we might write

\[
p(M,q,P,z,c,t,m,\sigma_m)
=
p(M,q,P,z)\,
p(c\mid z)\,
p(t\mid c)\,
p(m\mid t,M,q,P,z,c)\,
p(\sigma_m\mid m,c),
\]

where \(c\) denotes survey context. This is only a schematic—not a documented
statement of the Crete simulator internals—but it demonstrates how scientific
assumptions become conditional-independence assumptions.

A graphical model is therefore useful before any fitting. It forces us to say
whether, for example, cadence depends on field, whether photometric uncertainty
depends on magnitude, and which variables are latent versus observed.

### Conditional independence

If

\[
X\perp Y\mid Z,
\]

then

\[
p(x,y\mid z)=p(x\mid z)p(y\mid z).
\]

Conditional independence is often more realistic than marginal independence.
Two observables can be associated only because both depend on redshift. Once
redshift is conditioned on, the association may weaken or disappear.

This idea will later motivate regression with covariates, hierarchical models
and causal caution in feature-importance analysis.

## 12. Bayes' theorem preview: likelihood is not posterior

From

\[
p(D,\theta)=p(D\mid\theta)p(\theta)
           =p(\theta\mid D)p(D),
\]

we obtain

\[
p(\theta\mid D)
=
\frac{p(D\mid\theta)p(\theta)}{p(D)}.
\]

At this stage, the important conceptual distinction is:

- \(p(D\mid\theta)\): **forward/data-generating** statement;
- \(p(\theta\mid D)\): **inverse/inferential** statement.

Confusing these two directions is one of the most common errors in scientific
reasoning.

For example, even if high mass ratio makes a lensing flare more probable,
\(P({\rm flare}\mid q)\) is not the same as \(P(q\mid{\rm flare})\).
The latter also depends on how common each \(q\) value is in the population.

## 13. Transforming probability densities

Suppose \(Y=g(X)\) is a one-to-one transformation. Then

\[
p_Y(y)
=
p_X(g^{-1}(y))
\left|
\frac{d g^{-1}(y)}{dy}
\right|.
\]

For \(Y=\log_{10}X\),

\[
X=10^Y,\qquad
\frac{dX}{dY}=\ln(10)\,10^Y.
\]

Thus a distribution that is uniform in \(X\) is not uniform in \(\log X\).
This matters whenever we describe a "flat" or "power-law" distribution.

In astronomy, log transformations are often scientifically natural because
mass, luminosity and timescale ranges are multiplicative and span orders of
magnitude. But the transformation must be stated explicitly.

In [ ]:
# Compare equal-width bins in M and logM
m = df["inputs__total_mass"].astype(float).to_numpy()

fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].hist(m,bins=30)
axes[0].set_xlabel("Mtot [solar masses]")
axes[0].set_ylabel("count")

axes[1].hist(np.log10(m),bins=30)
axes[1].set_xlabel("log10 Mtot")
axes[1].set_ylabel("count")
plt.tight_layout()
plt.show()

## 14. Covariance matrices and multivariate thinking

For a random vector \(\mathbf X=(X_1,\ldots,X_p)^T\), the covariance matrix has
elements

\[
\Sigma_{ij}={\rm Cov}(X_i,X_j).
\]

The diagonal contains variances; off-diagonal terms contain covariances.
Covariance matrices are fundamental to multivariate Gaussian models, Gaussian
processes, PCA and uncertainty propagation.

But a covariance matrix captures only second-order linear dependence. If the
distribution is strongly nonlinear, multimodal or bounded, it is incomplete.

A useful Week-1 exercise is to compare Pearson and Spearman matrices after
physically motivated transformations such as \(\log M\) and \(\log P\).

## 15. Error propagation with covariance

For a differentiable derived quantity \(g(\mathbf x)\), first-order error
propagation gives

\[
{\rm Var}[g(\mathbf x)]
\approx
\nabla g^T\Sigma\nabla g.
\]

For two variables,

\[
\sigma_g^2
\approx
\left(\frac{\partial g}{\partial x}\right)^2\sigma_x^2
+
\left(\frac{\partial g}{\partial y}\right)^2\sigma_y^2
+
2\frac{\partial g}{\partial x}
\frac{\partial g}{\partial y}
{\rm Cov}(x,y).
\]

Dropping covariance terms when measurements are correlated can substantially
misestimate uncertainty. Monte Carlo propagation is often easier for nonlinear
transformations.

In [ ]:
# Correlated uncertainty propagation demonstration
mean = np.array([10.0, 4.0])
cov = np.array([[1.0, 0.6],[0.6, 0.64]])
draw = rng.multivariate_normal(mean,cov,size=150_000)
ratio = draw[:,0]/draw[:,1]

cov_ind=np.diag(np.diag(cov))
draw_ind=rng.multivariate_normal(mean,cov_ind,size=150_000)
ratio_ind=draw_ind[:,0]/draw_ind[:,1]

print("correlated 68%:",np.quantile(ratio,[.16,.84]))
print("independent 68%:",np.quantile(ratio_ind,[.16,.84]))

## 16. Robust statistics and outliers

The mean is sensitive to extreme values. The median is much more robust. A
common robust scale estimate is

\[
{\rm MAD}
=
{\rm median}(|x_i-{\rm median}(x)|).
\]

For Gaussian data, \(1.4826\,{\rm MAD}\) approximately estimates the standard
deviation.

Outliers can be bad measurements, unmodelled systematics or genuinely rare
objects. Robust statistics reduce their influence but do not decide which
interpretation is correct. An outlier should be investigated, not automatically
deleted.

## 17. Week-1 mini case study: write the question before the code

Consider:

> "Are short-period binaries more eccentric in the supplied simulations?"

A poor workflow is to plot period versus eccentricity, notice a pattern, choose
a convenient threshold and report a test.

A stronger workflow is to define whether the question concerns correlation, a
conditional mean, a population difference or prediction; choose transformations;
define "short-period" before confirmatory testing; identify context variables;
choose a statistic and uncertainty procedure; and reserve validation data if
the hypothesis arose through exploration.

The same data support many statistical questions. Precision starts with stating
which one is being answered.

## 18. End-of-lecture board exercise

Draw a directed generative diagram for

\[
(M,q,P,e,z,\mathrm{field})
\rightarrow
\text{latent variability}
\rightarrow
(t,m,\sigma_m)
\rightarrow
\text{features}
\rightarrow
\widehat P.
\]

Mark variables available in blinded data, variables available only as simulator
truth, where measurement uncertainty enters, where cadence enters, and where
leakage could occur.

This closes Week 1: probability is not a list of distributions; it is a language
for explicitly describing the scientific data-generating process.